## Exploring META data and Generating Subfolders for FB_INF project
### Targeted Modulation Version (GenV6)

#### LOADING THE NECESSARY LIBRARIES TO EXPLORE THE METADATA GENERATED

In [1]:

%load_ext autoreload
%autoreload 2

from wormholes import *
from wormholes.perturb import *
from wormholes.tools.triplets_vis_tools import *
import matplotlib.pyplot as plt
import matplotlib.image as img
from argparse import Namespace
import os
import xarray as xr #this is to be able to read the .nc type of file
import shutil
import random 
import pandas as pd

In [2]:
# Set the HOME environment variable to your current working directory
os.environ["HOME"] = "/project/3018078.01/Gaziv/Wormholes_FB"

# Now the `os.path.expanduser("~")` will resolve to this directory
print(os.path.expanduser("~")) 

/project/3018078.01/Gaziv/Wormholes_FB


This next part of the script is loading the meta data file and does some exploration of the variables that have been defined there. 

In [3]:
#this is the directory where they will find the images (change as you deem necessary)

exp_root = f"{PROJECT_ROOT}/results/cache/gen_v6"

# Load the metadata file
metadata = xr.open_dataset(f"{exp_root}/meta.nc")

# Print metadata structure
# print(metadata)

# List variables in the dataset
print(metadata.variables)

Frozen({'model_name': <xarray.Variable (image_id: 46200)>
array(['resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0', ...,
       'resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0'], dtype=object), 'budget': <xarray.Variable (image_id: 46200)>
array([ 0.,  0.,  0., ..., 30., 30., 30.]), 'n_iter': <xarray.Variable (image_id: 46200)>
array([   0,    0,    0, ..., 2000, 2000, 2000]), 'step_size': <xarray.Variable (image_id: 46200)>
array([0., 0., 0., ..., 2., 2., 2.]), 'interp_alpha': <xarray.Variable (image_id: 46200)>
array([nan, nan, nan, ..., nan, nan, nan]), 'target_class_name': <xarray.Variable (image_id: 46200)>
array(['turtle', 'lizard', 'bird', ..., 'insect', 'rabbit', 'gazelle'],
      dtype=object), 'orig_class_name': <xarray.Variable (image_id: 46200)>
array(['OOD-frog', 'OOD-frog', 'OOD-frog', ..., 'OOD-primate', 'OOD-primate',
     

I am now transforming the .nc file into a Pandas dataframe, as this is the type of structure I am familiar with and allows me to do the selection of relevant information I need for my experiment

In [4]:
#Depending on the size of the meta.nc file. The kernel might get overwhemled when trying to trasnform it to a pandas dataframe. 
#Therefore, the next couple of code lines are to only pass the information that is relevant (reducing the size of df)
selected_vars = ["model_name", "budget", "image_id",'class_index', 'pred_logit',
                 'model_subject_name', 'orig_class_name', 'orig_name', 'target_class_name']
df = metadata[selected_vars].to_dataframe().reset_index()

# Convert the dataset to a Pandas DataFrame (if I want to have the whole dataframe, I can use this)
# df = metadata.to_dataframe().reset_index()

# Inspect the metadata structure
print(df.head())

# Inspect the columns in the Pandas Dataframe
print(df.columns)

#information about the data in the columns
df.info(verbose=False)

                               image_id  class_index  \
0  65c445b6a56020704234dcb9ed34f4b3.png            0   
1  65c445b6a56020704234dcb9ed34f4b3.png            0   
2  65c445b6a56020704234dcb9ed34f4b3.png            0   
3  65c445b6a56020704234dcb9ed34f4b3.png            0   
4  65c445b6a56020704234dcb9ed34f4b3.png            0   

                      model_subject_name                         model_name  \
0  resnet50_robust_mapped_RIN_l2_10_0_v2  resnet50_robust_mapped_RIN_l2_3_0   
1   resnet50_robust_mapped_RIN_l2_1_0_v2  resnet50_robust_mapped_RIN_l2_3_0   
2      resnet50_robust_mapped_RIN_l2_3_0  resnet50_robust_mapped_RIN_l2_3_0   
3   resnet50_robust_mapped_RIN_l2_3_0_v2  resnet50_robust_mapped_RIN_l2_3_0   
4         resnet50_vanilla_mapped_RIN_v2  resnet50_robust_mapped_RIN_l2_3_0   

   budget  pred_logit orig_class_name                 orig_name  \
0     0.0    2.090741        OOD-frog  OOD/frog/frog_00022.JPEG   
1     0.0    2.415928        OOD-frog  OOD/frog/frog_0

I am now filtering the dataframe so it only includes the parameters that are relevant for my experiment. 
With the filtered data frame I can then create folders with the relevant images for the experiment (with the correct names)

NOTE: model_name is the variable used to define which model is being used to generate the adversarial images. model_subject_name is all the other models that will be tested on the adversarial images generated by model_name and compare their performances. 

In [5]:
#The following lines of code are ways of probing the df. Not necessary, but useful

# print(df['model_name'].unique()) # displays all the networks available to extract images from
# print(df['model_subject_name'].unique()) #the v2 versions of the networks are just initialized using different seeds
# print(df['budget'].unique()) #all the budget options I can select images from


# Apply filtering (as model subject name makes a double entry on all meta data)
filtered_df = df[(df['model_subject_name'] == 'resnet50_robust_mapped_RIN_l2_3_0')] 
# print(filtered_df.head())
# print(filtered_df.columns)
# print(filtered_df['model_name'])
# print(len(filtered_df['image_id'].unique()))
# filtered_df.shape
print(filtered_df['class_index'].unique())

#Code to visualize any image we want on the dataset
# im = img.imread(f"{exp_root}/images/30a0f974d91cf438f69c726c9434678b.png")
# plt.imshow(im)


[ 0  1  2  3  4  5  6  7  8  9 10 11]


Now, I want to generate code that creates a new folder with all my perturbed images. 
I will devise a specific form of naming them that will make it easier for me to use in the actual experiment. 

naming convention = orig(origclass)_target(targetclass)_budget_imgnumber

I will create code to generate a folder that includes the images for a specific budget. 

As we have not decided if we are going with 10 or with 12 classes, I will create either two functions or one function with a parameter choice to decide to create a dataset that already satisfies the needs of the experiment.


In [74]:
#Creating a new folder with images per class, renamed and (nearly) ready for me to use for the experiments 

def stimulus_set_folder_per_budget (class_num, df, budget, output_folder_root = "/project/3018078.01/FB_INF/stimulus_set", seed = 0):

    #create a folder in the directory where I will store the images
    folder_dir_path = os.path.join(output_folder_root, str(budget))
    # Create the folder if it doesn't exist
    os.makedirs(folder_dir_path, exist_ok=True)

    #filter out dataframe so it only includes the desired budget

    if class_num == 10:
        df = df[(df['budget'] == budget) & (~df["orig_class_name"].isin(["OOD-cat", "OOD-gazelle"])) 
        & (~df["target_class_name"].isin(["cat", "gazelle"]))]

        samples_per_class = 36
        targets_per_sample = 4
    
    elif class_num == 12:
        df = df[(df['budget'] == budget)] 
        samples_per_class = 22
        targets_per_sample = 2

    else:
        raise ValueError(f"Argument class_num must be one of the two options: 10 or 12.")


    # Initialize list to store selected rows
    selected_rows = []

    # Loop over each original class to select only a certain amount of classes
    random.seed(seed)  # Ensure reproducibility before the loop
    for _, class_df in df.groupby("orig_class_name"):
        unique_images = class_df["orig_name"].unique()

        if len(unique_images) < samples_per_class:
            raise ValueError(f"Class {class_df['orig_class_name'].iloc[0]} has fewer than 22 images available.")

        selected_origimage_ids = random.sample(list(unique_images), samples_per_class)  # Select number unique images
        selected_rows.append(class_df[class_df["orig_name"].isin(selected_origimage_ids)])

    # Combine all selected rows into a new dataframe
    selected_df = pd.concat(selected_rows, ignore_index=True)

    
    selected_target_orig_pair_rows =[]

    
    for orig_class_name, orig_class_df in selected_df.groupby("orig_class_name"):
        final_selected_rows = []
        selected_orig_name = set()
        for target_class_name, target_df in orig_class_df.groupby("target_class_name"):
        
            # Remove images that are already selected
            target_df = target_df[~target_df["orig_name"].isin(selected_orig_name)]
            orig_class_df = orig_class_df[~orig_class_df["orig_name"].isin(selected_orig_name)]

            # Ensure we have at least unique images for this target class
            if len(target_df) < targets_per_sample:
                raise ValueError(f"Target class '{target_class_name}' in orig_class '{orig_class_name}' has fewer than number unique distractor images.")
            # Randomly select number unique distractor images
            # selected_target_images = target_df.sample(n=targets_per_sample, random_state=seed)
            selected_target_images = target_df.drop_duplicates(subset="orig_name").sample(n=targets_per_sample, random_state=seed)

            
            # Add selected image_ids to the set to prevent reuse
            selected_orig_name.update(selected_target_images["orig_name"])

            final_selected_rows.append(selected_target_images)

        # Combine selected images for this orig_class_name
        selected_target_orig_pair_rows.append(pd.concat(final_selected_rows, ignore_index=True))

    # Combine all selected rows into a new dataframe
    final_df = pd.concat(selected_target_orig_pair_rows, ignore_index=True)

    #copying the 360 images
    for orig_class_name, class_df in final_df.groupby("orig_class_name"):
        for i, row in enumerate(class_df.itertuples(index=False)):
            file_name = f"orig-{row.orig_class_name}_target-{row.target_class_name}_{row.budget}_{i+1}.png"
            destination_path = os.path.join(folder_dir_path, file_name)
            source_path = os.path.join(exp_root, f'images/{row.image_id}')
            shutil.copy(source_path, destination_path)
        

tryout = stimulus_set_folder_per_budget(class_num = 10, df = filtered_df, budget = 12.5, seed=1)


# Check if it worked:

# print(tryout["orig_class_name"].value_counts())  # Should be 36 per orig_class
# print(tryout.groupby(["orig_class_name", "target_class_name"])["orig_name"].nunique())  # Should be 4 per target_class
# print(tryout["orig_name"].nunique())  # Should not have duplicates! 
    